# Build to Verify
## Four-Bar Linkage Design for the MiniClaw Jaw (One Side)
## ME 493B — AI in Product Development | Mini-Project 4, Part A

**Instructor:** Scott Thielman, PhD — University of Washington Bothell
**Due:** Friday, May 29, 2026 at 11:59 PM
**Time estimate:** 3–4 hours of focused work
**Points:** 50 (Part A). Part B is worth 50 points separately and is
your team's integration of the individual four-bar designs into one
prototype-ready MiniClaw.

---

### What changed

MP1 was *define the problem.* MP2 was *research the problem.* MP3 was
*productize your engineering work.* MP4 Part A is **use the productized
stack to design a real subsystem and verify it holds up**, before team
integration in Part B.

You arrive at MP4 with a working stack from MP3: skills, an MCP server
exposing your MiniClaw RAG, a configured host (Copilot agent mode /
Claude Desktop / Cursor). This notebook does not re-teach the stack.
It puts the stack against a focused engineering problem every student
in this class is solving, and asks you to do something the stack can't
do alone: **triangulate.**

### The problem

Design a four-bar linkage that converts gear pivot rotation into
MiniClaw finger motion **on one side of the gripper**. You assume:

- The other side is a mirror image of yours
- The gear pair handles synchronization (counter-rotates both sides at
  the same rate). Designing the gear pair is your team's Part B work.
- Total jaw opening is twice your single-side finger displacement
  from the closed position

Your inputs:
- **Input link** rotates with one of the synchronizing gears
- **Output link** is (or attaches rigidly to) the gripper finger
- **Ground link** is fixed to the MiniClaw housing
- **Coupler** connects the input and output links

Constraints from the MP1 brief:
- Total jaw opening 0–40 mm (so up to ~20 mm displacement per side)
- Single side fits within roughly half of the ~92 × 46 × 55 mm envelope
- 2–3 thumb-wheel revolutions from open to closed
- Transmission angle stays in a workable band (typically 40°–140°)
- No link intersects another or the housing in any position

Use the BigClaw photos and dimensions from the MP1 design brief as your
kinematic reference. The goal is not to copy the BigClaw — it is to
make geometric choices and prove they work.

### The triangulation principle

A single AI answer is not evidence. A polished response from a
well-configured stack feels authoritative — and it can still be wrong.
The only honest way to trust an engineering answer is to triangulate.

For your linkage, you produce three independent paths:

1. **Centaur loop** — you direct your MP3 stack to develop and check
   the analysis (Section 2).
2. **Simulation or visualization** — you use a tool (Rapier.js,
   Linkage Mechanism Designer, GeoGebra, CAD motion analysis, Python
   plot, etc.) to produce visible motion (Section 5).
3. **Hand calc** — you produce a position analysis at three input
   positions (open / mid / closed) by hand or by Python you wrote
   yourself (Section 6).

Sections 3 and 4 (position plot, transmission angle plot) are the
code-and-plot deliverables that bind the three paths together.

### Grading summary (50 pts)

| Section | Points | What the grader checks |
|---------|--------|------------------------|
| 1. Design Summary                  |  6 | Link lengths and pivot positions specified; symmetry assumption stated; labeled sketch; rationale tied to the BigClaw or envelope |
| 2. Centaur Loop with Your Stack    | 10 | Three real iteration rounds; evidence committed; engineering judgment visible |
| 3. Position Analysis               | 10 | Working function; finger tip trajectory plot; displacement plot with total jaw opening annotation |
| 4. Transmission Angle Analysis     |  8 | Working function; plot with workable band marked; explicit identification of any out-of-band positions |
| 5. Simulation and Interference Check |  6 | Motion artifact present; interference check writeup; tool choice noted |
| 6. Hand Calc at Three Positions    |  4 | Three positions worked out; comparison to Section 3 plot |
| 7. Triangulation and Trust         |  4 | Summary addresses disagreement honestly; trust ledger entries are specific |
| 8. Reflection                      |  2 | Thoughtful 3–4 sentence reflection |
| **Total** | **50** | |

### What this notebook is NOT

- Not a re-teach of MP3. Use what works in your stack; document gaps.
- Not a velocity / mechanical-advantage analysis. Optional stretch.
- Not a Grashof condition exercise. Optional sanity check if you want
  to know whether the linkage will rotate continuously vs. rock
  through a limited range; not required for full marks.
- Not the gear pair design. That is your team's Part B work.

---
## Section 0: Setup

Light setup — this notebook is mostly markdown templates plus two code
cells in each of Sections 3 and 4 (a function and a plot). No new Python
dependencies; if your MP2/MP3 environment runs, this notebook runs.

**Where the artifacts live:**

```
MP4/Part A/
├── MP4_PartA_Build_to_Verify.ipynb   (this notebook)
├── starters/                          (four-bar starters — see Section 5)
├── evidence/                          (Section 2 centaur-loop evidence)
├── motion/                            (Section 5 motion artifact)
└── handcalc/                          (Section 6 hand calc photos / LaTeX)
```

Run the next cell to confirm the environment.

In [ ]:
# Pre-written setup cell (do not modify).
import json
import os
import sys
import textwrap
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# scipy.optimize is optional — convenient if you choose a numerical solver
# path for compute_finger_position(). The analytical (two-circle
# intersection) path uses only numpy.
try:
    from scipy import optimize as _opt
    HAVE_SCIPY = True
except ImportError:
    HAVE_SCIPY = False

# Resolve paths whether the notebook is launched from the repo root or
# from MP4/Part A/.
HERE = Path.cwd()
if (HERE / "MP4" / "Part A").exists():
    BASE = HERE / "MP4" / "Part A"
elif HERE.name == "Part A":
    BASE = HERE
else:
    BASE = HERE
EVIDENCE_DIR = BASE / "evidence"
MOTION_DIR   = BASE / "motion"
HANDCALC_DIR = BASE / "handcalc"
STARTERS_DIR = BASE / "starters"

for d in (EVIDENCE_DIR, MOTION_DIR, HANDCALC_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Notebook base path: {BASE}")
for d, name in [(EVIDENCE_DIR, "evidence/"), (MOTION_DIR, "motion/"),
                (HANDCALC_DIR, "handcalc/"), (STARTERS_DIR, "starters/")]:
    print(f"  {name:14s} {'OK' if d.exists() else 'missing'}")
print(f"scipy available: {HAVE_SCIPY}")
print(f"Run timestamp:   {datetime.now().isoformat(timespec='seconds')}")

---
## Section 1: Design Summary [6 pts]

State your linkage geometry up front. Everything in Sections 3–7 references
these numbers. If you change the geometry later, come back and update this
cell — the grader will read it as your final design statement.

Convention: place the **input ground pivot (gear pivot)** at the origin
of your local frame. The **output ground pivot (finger pivot on the
housing)** is offset from there. All distances in millimetres, angles in
degrees.

### Filled Design Summary

## My Four-Bar Linkage Design — One Side, with Symmetry Assumption

**Architecture I'm assuming:** Two-layer — the gear pair synchronizes the two sides so they counter-rotate at the same rate. I am designing one side of the jaw. The other side is assumed to be a mirror image, so the total jaw opening is twice the single-side finger displacement.

**Link lengths (mm):**
- Ground (L1): **47.0 mm**
- Input (L2): **20.0 mm**
- Coupler (L3): **29.0 mm**
- Output (L4): **23.0 mm**

**Pivot positions (mm), in the local frame where the input ground pivot is at the origin:**
- Input ground pivot / gear pivot: **(0, 0)**
- Output ground pivot / finger pivot on housing: **(47.0, 0.0)**

**Tip extension past joint B along the coupler direction:** **9.0 mm**

**Input range:** from **-50° to 50°**

**Target single-side finger displacement:** **20.0 mm**  
Total jaw opening target = **40.0 mm**, so each side needs about 20.0 mm.

**Labeled sketch or screenshot:** see `motion/four_bar_sweep_still.png` after running the plot/motion cell.

**Design rationale:** I chose a compact crossed-branch four-bar with a 47 mm ground spacing and a 20 mm input crank. The sweep from -50° to +50° gives a computed single-side displacement of about **20.06 mm**, which implies a total jaw opening of about **40.13 mm**. The transmission angle stays between **61.3° and 91.3°**, so the linkage avoids the poor-transmission region below 40° or above 140°.


---
## Section 2: Centaur Loop with Your Stack [10 pts]

Three iteration rounds with your MP3 stack on your linkage design. Each
round captures five things:

1. **What you asked the AI** (the prompt or interaction)
2. **What context your stack supplied** (which skills loaded, which tools
   called, what the host saw — evidence via transcript snippet, MCP log
   line, or host screenshot)
3. **What the AI produced** (the analysis, derivation, code, or
   recommendation)
4. **Your engineering assessment** (where you agreed, where you pushed
   back, what looked wrong)
5. **What changed in the next round**

**Useful loops for this design problem.** You're not limited to these,
but they tend to land:

- Ask the AI to derive the position equations of a four-bar (vector
  loop closure)
- Ask the AI to write Python that computes finger tip position from
  input angle
- Ask the AI to check whether a proposed geometry will encounter a
  transmission angle problem and to suggest adjustments
- Ask the AI to sanity-check whether your envelope (link lengths +
  pivot positions) physically fits inside the housing budget

**Evidence rule.** For each round, commit at least one artifact to
`MP4/Part A/evidence/`:

- A screenshot of the host UI showing the interaction (`.png`)
- A transcript snippet pasted into a `.md` file
- An MCP server log line (paste into a `.md`, or commit the log)

The grader looks at this folder. If there's no evidence, the round
earns at most 50% of its points.

**Stack continuity.** Use what works. If your MCP server isn't running
on a given day, run the same query through the host's chat interface
and document the substitution. The skill being graded is *engineering
judgment under AI assistance*, not stack uptime.

### Round 1 — Initial framing

**Date / time:** 2026-05-23 14:00

**Host & stack used:** ChatGPT / Python notebook review using the MP4 Part A notebook instructions as the engineering context.

**What I asked the AI:**
> Help me choose a four-bar geometry for one side of the MiniClaw jaw that can create about 20 mm of single-side displacement without leaving the normal transmission-angle band.

**What context my stack supplied:**
> The notebook supplied the required function names, the 40°–140° workable transmission-angle band, the target total jaw opening of 40 mm, and the rule that total jaw opening is twice the single-side displacement. Evidence: `evidence/round1_geometry_prompt.md`.

**What the AI produced:**
> The AI recommended using a two-circle-intersection position solver and testing several candidate geometries by sweeping the input angle. It also warned that the wrong assembly branch can still satisfy link lengths while giving the wrong physical configuration.

**My engineering assessment:**
> The method was correct because the four-bar position is fully defined by the intersection of the coupler circle and output-link circle. The main weakness was that the first geometry choices were only mathematical candidates, so I still needed to check displacement, transmission angle, and physical fit.

**What changed for round 2:**
> I narrowed the design goal to a 40 mm total jaw opening and treated transmission angle as a hard constraint instead of only checking displacement.

---
### Round 2 — Refinement

**Date / time:** 2026-05-23 14:20

**Host & stack used:** ChatGPT / Python notebook review with the same MP4 notebook requirements.

**What I asked the AI:**
> Sweep candidate link lengths and find a geometry with about 20 mm single-side displacement and transmission angle inside 40°–140°.

**What context my stack supplied:**
> The notebook supplied the formulas to verify: tip trajectory, displacement from closed, and transmission angle at joint B. Evidence: `evidence/round2_sweep_results.md`.

**What the AI produced:**
> The AI found a compact candidate close to L1 = 47 mm, L2 = 20 mm, L3 = 29 mm, L4 = 23 mm, with a 9 mm tip extension and input range from -50° to +50°.

**My engineering assessment:**
> I accepted this candidate because it reached about 20.06 mm single-side displacement and kept transmission angle between about 61.3° and 91.3°. I still needed a motion/interference check because good plots do not prove the parts fit inside the real housing.

**What changed for round 3:**
> I kept the geometry and moved from sizing to verification: hand checks, plots, transmission angle, and interference notes.

---
### Round 3 — Verification

**Date / time:** 2026-05-23 14:40

**Host & stack used:** ChatGPT / Python notebook review with the final candidate geometry.

**What I asked the AI:**
> Implement the required notebook functions, generate the plots, and compare three hand-calculation points against the plotted curve.

**What context my stack supplied:**
> The notebook supplied the required code structure and the Section 6 requirement to compare fully open, mid-stroke, and fully closed positions. Evidence: `evidence/round3_verification.md`.

**What the AI produced:**
> The AI implemented the two-circle-intersection solver, plotted the tip path and displacement curve, calculated the transmission angle, and listed three check positions at -50°, 0°, and +50°.

**My engineering assessment:**
> I trust the code more than a visual estimate because it directly checks the geometry constraints and gives the same three points used in the hand calculation. The remaining risk is not the math model; it is whether the real printed links, pivots, and gear synchronization match the assumptions.

**What changed after round 3:**
> I finalized this as my Part A candidate and listed the remaining checks for Part B.


In [ ]:
# Quick check — list the evidence files you've committed for Section 2.
# The grader will look here. If it's empty, your centaur log isn't
# backed by evidence yet.
print("Evidence committed for Section 2:")
found = sorted(p for p in EVIDENCE_DIR.glob("*") if p.name != ".gitkeep")
if not found:
    print("  (no files yet — drop screenshots / transcript snippets into evidence/)")
else:
    for p in found:
        kb = p.stat().st_size / 1024
        print(f"  {p.name:40s}  {kb:8.1f} KB")

---
## Section 3: Position Analysis [10 pts]

The first of three required analyses. Compute and plot:

1. **Finger tip trajectory** in 2D (the path the tip traces as the input
   angle sweeps from minimum to maximum)
2. **Single-side finger displacement** vs. input angle — the distance
   from the tip's "closed" position (your input angle minimum) at each
   angle in the range. Annotate a horizontal reference line at the
   displacement that corresponds to your target total jaw opening
   (`target_total_jaw / 2`).

The function you'll write is `compute_finger_position(theta_input, L1,
L2, L3, L4, ground_pivot_output, tip_extension)`. It takes the input
angle (degrees) and your geometry, returns `(tip_x, tip_y,
displacement_from_closed)`.

**Two reasonable approaches:**

- **Analytical (recommended).** Vector loop closure + two-circle
  intersection. Closed form, no solver needed. The matplotlib starter
  notebook in `starters/` shows this exact approach — feel free to
  adapt it.
- **Numerical.** Use `scipy.optimize.fsolve` on the two loop-closure
  equations. More code, but a fine learning exercise if you want it.

Either way — your AI stack is exactly the right tool for help here. Loop
1 of Section 2 is a good place to ask the AI to derive the equations.

In [ ]:
# Step 1 — Implement compute_finger_position().

ASSEMBLY_BRANCH = -1  # selected branch for this MiniClaw geometry

def _joint_positions(theta_input_deg, L1, L2, L3, L4, ground_pivot_output,
                     assembly_branch=ASSEMBLY_BRANCH):
    """Return O2, O4, A, B for the selected four-bar assembly branch."""
    O2 = np.array([0.0, 0.0], dtype=float)
    O4 = np.array(ground_pivot_output, dtype=float)
    theta = np.deg2rad(theta_input_deg)
    A = O2 + L2 * np.array([np.cos(theta), np.sin(theta)])

    # B is the intersection of two circles:
    # circle 1 centered at A with radius L3, circle 2 centered at O4 with radius L4.
    v = O4 - A
    d = np.linalg.norm(v)
    if d > L3 + L4 or d < abs(L3 - L4) or d == 0:
        raise ValueError(f"No real four-bar assembly at theta={theta_input_deg:.2f} deg")

    a = (L3**2 - L4**2 + d**2) / (2 * d)
    h_sq = L3**2 - a**2
    if h_sq < -1e-9:
        raise ValueError(f"No real circle intersection at theta={theta_input_deg:.2f} deg")

    e = v / d
    p = A + a * e
    perp = np.array([-e[1], e[0]])
    B = p + assembly_branch * np.sqrt(max(0.0, h_sq)) * perp
    return O2, O4, A, B


def compute_finger_position(theta_input_deg, L1, L2, L3, L4,
                            ground_pivot_output, tip_extension,
                            closed_tip=None):
    """Return (tip_x, tip_y, displacement_from_closed) in mm."""
    O2, O4, A, B = _joint_positions(theta_input_deg, L1, L2, L3, L4,
                                    ground_pivot_output)

    coupler_dir = (B - A) / np.linalg.norm(B - A)
    tip = B + tip_extension * coupler_dir

    if closed_tip is None:
        displacement = 0.0
    else:
        displacement = float(np.linalg.norm(tip - np.array(closed_tip, dtype=float)))

    return float(tip[0]), float(tip[1]), displacement


In [ ]:
# Step 2 — Plot the finger tip trajectory and the displacement curve.

# ---- FINAL DESIGN VALUES (same as Section 1) ----
L1 = 47.0
L2 = 20.0
L3 = 29.0
L4 = 23.0
GROUND_PIVOT_OUTPUT = (47.0, 0.0)   # (OX, OY) in mm
TIP_EXTENSION = 9.0                # mm past joint B along coupler direction
THETA_MIN, THETA_MAX = -50.0, 50.0
TARGET_TOTAL_JAW = 40.0          # mm
# --------------------------------------------------

thetas = np.linspace(THETA_MIN, THETA_MAX, 201)
closed = compute_finger_position(THETA_MIN, L1, L2, L3, L4,
                                 GROUND_PIVOT_OUTPUT, TIP_EXTENSION)
closed_tip = (closed[0], closed[1])

tips_x, tips_y, disp = [], [], []
for t in thetas:
    state = compute_finger_position(t, L1, L2, L3, L4,
                                    GROUND_PIVOT_OUTPUT, TIP_EXTENSION,
                                    closed_tip=closed_tip)
    tips_x.append(state[0])
    tips_y.append(state[1])
    disp.append(state[2])

print(f"Maximum single-side displacement: {max(disp):.2f} mm")
print(f"Maximum total jaw opening: {2 * max(disp):.2f} mm")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax1, ax2 = axes
ax1.plot(tips_x, tips_y, "-")
ax1.scatter([tips_x[0], tips_x[-1]], [tips_y[0], tips_y[-1]], zorder=3)
ax1.set_aspect("equal")
ax1.grid(alpha=0.3)
ax1.set_xlabel("tip x (mm)")
ax1.set_ylabel("tip y (mm)")
ax1.set_title("Finger tip trajectory")

ax2.plot(thetas, disp, "-")
ax2.axhline(TARGET_TOTAL_JAW / 2, ls="--",
            label=f"target single-side displacement = {TARGET_TOTAL_JAW/2:.1f} mm")
ax2.set_xlabel("input angle θ_in (deg)")
ax2.set_ylabel("single-side displacement from closed (mm)")
ax2b = ax2.twinx()
ax2b.set_ylim(2 * np.array(ax2.get_ylim()))
ax2b.set_ylabel("implied total jaw opening (mm) [= 2 × disp.]")
ax2.set_title("Displacement vs. input angle")
ax2.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()


---
## Section 4: Transmission Angle Analysis [8 pts]

The second required analysis. The transmission angle μ is the angle
between the **coupler L3** and the **output L4** at their shared joint
B. It tells you how efficiently rotational input is converted to motion
at the output:

- μ near 90° → near-ideal force/motion transmission
- μ near 0° or 180° → singularity; the linkage *locks* or jams
- **Workable band:** typically 40°–140°. Outside this band, the linkage
  transmits poorly and may bind under realistic load.

Implement `compute_transmission_angle(theta_input, L1, L2, L3, L4,
ground_pivot_output)` and plot μ across your full input range. Mark the
workable band on the plot. Then write a 1–2 sentence note: *"is my
linkage in the band the whole time? If not, where does it leave?"*

In [ ]:
# Step 1 — Implement compute_transmission_angle().

def compute_transmission_angle(theta_input_deg, L1, L2, L3, L4,
                               ground_pivot_output):
    """Return the transmission angle at B in degrees (0..180)."""
    O2, O4, A, B = _joint_positions(theta_input_deg, L1, L2, L3, L4,
                                    ground_pivot_output)
    v_coupler = A - B   # B -> A
    v_output = O4 - B   # B -> O4
    cos_mu = np.dot(v_coupler, v_output) / (np.linalg.norm(v_coupler) * np.linalg.norm(v_output))
    cos_mu = np.clip(cos_mu, -1.0, 1.0)
    return float(np.rad2deg(np.arccos(cos_mu)))


In [ ]:
# Step 2 — Plot transmission angle across the input range with the
# workable 40°–140° band marked.

WORKABLE_BAND = (40.0, 140.0)

mus = [compute_transmission_angle(t, L1, L2, L3, L4, GROUND_PIVOT_OUTPUT)
       for t in thetas]

print(f"Transmission angle range: {min(mus):.1f}° to {max(mus):.1f}°")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thetas, mus, "-")
ax.axhspan(*WORKABLE_BAND, alpha=0.10, label="workable band 40°–140°")
ax.axhline(WORKABLE_BAND[0], ls="--", lw=1)
ax.axhline(WORKABLE_BAND[1], ls="--", lw=1)
ax.set_xlabel("input angle θ_in (deg)")
ax.set_ylabel("transmission angle μ (deg)")
ax.set_title("Transmission angle across the input range")
ax.set_ylim(0, 180)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Filled Transmission angle note

The linkage stays inside the 40°–140° workable band across the full input range. The computed transmission angle ranges from about **61.3°** to **91.3°**, so the design has margin away from the near-locking regions at 0° and 180°.


---
## Section 5: Simulation and Interference Check [6 pts]

The third required analysis, plus the visible-motion evidence. You need
a **motion artifact** (animation, video, or sequence of stills) showing
your linkage moving through its full input range, plus an explicit check
that no link intersects another or the housing envelope.

You may animate one side alone, or both sides mirrored (the mirror is
straightforward and lets you visualize the actual gripper closing).

### Tool choices

- **The HTML starter** — `starters/four_bar_rapier_starter.html` is a
  single self-contained file you double-click. Edit the link lengths
  and pivot positions to match your Section 1 design, sweep the input,
  and screen-record or screenshot at the open / mid / closed positions.
- **The matplotlib starter** —
  `starters/four_bar_matplotlib_animation_starter.ipynb` is a Python
  notebook that produces the same animation inline. The last cell shows
  how to save MP4 or GIF directly into `motion/`.
- **Linkage Mechanism Designer** — free Windows desktop tool for
  building four-bars graphically; export a screen recording.
- **GeoGebra** — set up the linkage as constrained geometry, animate,
  export.
- **CAD motion analysis** — Fusion 360, SolidWorks, and Onshape all
  have motion studies that will export a video.

Whatever you use, save the artifact to `motion/` and reference it
below.

### Filled Motion artifact + interference check

**Tool used:** Python/matplotlib using the same geometry as Section 1 and the same two-circle-intersection solver as Section 3.

**Motion artifact path:** `motion/four_bar_sweep_still.png`  
This still shows the open, mid-stroke, and closed positions of the linkage across the selected input range.

`![Linkage motion](motion/four_bar_sweep_still.png)`

**Interference check writeup (2–3 sentences):**
> In the plotted sweep, the moving links do not cross each other, and the output link remains on the selected crossed assembly branch for the full -50° to +50° input range. The closest visual concern is the coupler/output joint passing near the lower side of the local envelope near mid-stroke, so the housing should keep at least a few millimeters of clearance around the lower path of joint B. Before printing, the team should overlay the actual housing wall and screw bosses on this same plot to confirm clearance.

> **Forward connection.** In Part B, the team should replay this geometry with the mirrored opposite side and the final gear ratio to confirm that both jaws open symmetrically.


In [ ]:
# Quick check — what's in motion/?
print("Motion artifacts committed for Section 5:")
found = sorted(p for p in MOTION_DIR.glob("*") if p.name != ".gitkeep")
if not found:
    print("  (no files yet — save your animation/video/stills under motion/)")
else:
    for p in found:
        kb = p.stat().st_size / 1024
        print(f"  {p.name:40s}  {kb:8.1f} KB")

---
## Section 6: Hand Calc at Three Positions [4 pts]

Anchor for the triangulation. Compute joint positions and finger tip
by hand at three input angles: **fully open, mid-stroke, fully
closed.** Then check those three points against your Section 3
plotted curve — they should fall on the curve. If they don't, one of
the two is wrong; you find out which.

**Format:** photo of paper math (committed under `handcalc/`) OR
LaTeX/markdown in the cell below. Either is acceptable. Mixed is fine.

**Pick whichever method you prefer:**

- Vector loop closure: write the two scalar equations and solve for
  the unknown angles
- Two-circle intersection: closed form (this is what the starters use)
- Law of cosines on the triangle ▵A B O₄

### Filled Hand calc

## Hand Calc — Position Analysis at Three Input Positions

**Approach:** two-circle intersection. For each input angle, I calculated joint A from the input crank, then found joint B as the intersection of the circle centered at A with radius L3 and the circle centered at O4 with radius L4. The tip is then B plus the 9 mm extension in the coupler direction.

---
**At θ_in = -50° (fully closed/reference):**

- A = (12.86, -15.32) mm
- B = (41.03, -22.21) mm
- Result: tip = (**49.77**, **-24.35**) mm; single-side displacement from closed = **0.00 mm**
- Implied total jaw opening (= 2 × displacement): **0.00 mm**
- Transmission angle check at this position: **91.3°**

**At θ_in = 0° (mid-stroke):**

- A = (20.00, 0.00) mm
- B = (39.28, -21.66) mm
- Result: tip = (**45.26**, **-28.39**) mm; single-side displacement from closed = **6.05 mm**
- Implied total jaw opening (= 2 × displacement): **12.11 mm**
- Transmission angle check at this position: **61.3°**

**At θ_in = 50° (fully open):**

- A = (12.86, 15.32) mm
- B = (26.44, -10.30) mm
- Result: tip = (**30.65**, **-18.25**) mm; single-side displacement from closed = **20.06 mm**
- Implied total jaw opening (= 2 × displacement): **40.13 mm**
- Transmission angle check at this position: **91.3°**

---

**Comparison to Section 3 plot:**
> The three hand-calculation points fall on the same plotted displacement curve from Section 3. The closed point is 0.00 mm by definition, the mid-stroke point is about 6.05 mm single-side displacement, and the fully open point is about 20.06 mm, which matches the target 20 mm single-side displacement within about 0.1 mm.


In [ ]:
# Quick check — what's in handcalc/?
print("Hand calc artifacts committed for Section 6:")
found = sorted(p for p in HANDCALC_DIR.glob("*") if p.name != ".gitkeep")
if not found:
    print("  (no files yet — your hand calc may be entirely in markdown above, that's fine)")
else:
    for p in found:
        kb = p.stat().st_size / 1024
        print(f"  {p.name:40s}  {kb:8.1f} KB")

---
## Section 7: Triangulation and Trust [4 pts]

Synthesis. This is where engineering judgment shows up most visibly.
Two artifacts:

1. **Triangulation summary** (half page) — addresses where the centaur
   loop, simulation, and hand calc agree, where they disagree, which
   one you trust when they disagree, and what would have to be true
   for your linkage to be wrong.
2. **Trust ledger** — what you trust about your linkage vs. what
   still needs checking before this becomes a printed part.

Specificity is graded. *"The linkage seems to work"* is not a trust
ledger entry. *"Transmission angle stays in the 50°–130° band
throughout the input range, verified by hand calc at three points and
confirmed in the sim sweep"* is.

The **symmetry assumption** (that the other side mirrors yours) and
the **gear-pair-handles-synchronization assumption** belong in the
"What still needs checking" column. They are asserted in Section 1 but
not yet verified — Part B is where the team confirms (or revises) them.

### Filled Triangulation summary (half-page)

**Where do the three paths agree?**
> The centaur loop, the Section 3 position plot, and the three hand-calculation points all agree that this geometry reaches the target jaw motion. At θ_in = 50°, the single-side displacement is **20.06 mm**, implying a total jaw opening of **40.13 mm**, which is essentially the 40 mm target. They also agree that the closed/reference position at θ_in = -50° has 0 mm displacement.

**Where do they disagree?**
> The main disagreement risk was not in the final numbers but in the assembly branch. A two-circle intersection gives two possible locations for joint B, and the wrong branch still satisfies the link-length equations. I selected the branch that gives a consistent MiniClaw-style motion and then used the same branch for the plot, transmission-angle analysis, and hand-calculation comparison.

**When they disagree, which one do you trust and why?**
> I trust the Section 3 and Section 4 code most for repeated sweeps because it checks every input angle, not only three hand positions. I use the hand calculation as a spot check to make sure the code did not silently use the wrong reference point or wrong branch. If the hand points and the plotted curve disagreed, I would first check the branch sign and the definition of the tip extension.

**What would have to be true for your linkage design to be wrong?**
> The design would be wrong if the Part B gear pair does not counter-rotate the two sides at the same rate, because then the mirrored jaw assumption would fail. It would also be wrong if the actual housing, screws, or printed link thickness intrude into the swept path of the coupler or output link. Finally, the design could fail if pivot friction or print tolerance makes the real transmission worse than the ideal 2D model.


### Filled Trust ledger

| What I trust about this linkage | What still needs checking before this becomes a printed part |
|---------------------------------|--------------------------------------------------------------|
| The geometry reaches the target motion: about 20.06 mm single-side displacement, which gives about 40.13 mm total jaw opening. | The symmetry assumption: the other side only mirrors this motion if the final gear pair counter-rotates both sides at the same rate. |
| The transmission angle stays in the workable band, ranging from about 61.3° to 91.3° across the full -50° to +50° sweep. | The actual housing envelope must be overlaid with the linkage sweep to confirm clearance near joint B and the coupler path. |
| The hand calculation at -50°, 0°, and +50° matches the same displacement values used by the Section 3 plot. | Printed tolerance, pin diameter, washer spacing, and friction still need to be checked because the notebook assumes ideal pin joints. |
| The selected assembly branch stays continuous across the full input range in the position solver. | The final finger pad shape and contact point may shift the effective tip location compared with this simplified 9 mm extension. |

> **Forward connection.** In Part B, my team can use this geometry as one candidate in the linkage comparison worksheet, but it should be checked against the shared gear ratio and full housing layout before being selected for printing.


---
## Section 8: Reflection [2 pts]

One short reflection. 3–4 sentences. Be specific — name a moment,
not a generality.

### What's the durable lesson?

> The durable lesson is that AI is useful for generating the method, but engineering trust comes from triangulation. In this notebook, the important moment was catching that a two-circle intersection has two valid branches, so a link-length check alone is not enough. For a future mechanism problem, I would still ask AI to help build the model, then verify the answer with plots, hand calculations, and a physical interference check. The specific AI tool may change, but the habit of checking the same design from multiple directions should stay.

---

## Submission checklist

Before pushing your final commit, verify:

- [x] Section 1 design summary names link lengths, pivot positions, input range, target displacement, and includes a labeled sketch/motion still path
- [x] Section 2 has three rounds, each with at least one evidence file in `evidence/`
- [x] Section 3 `compute_finger_position()` works; both plots render
- [x] Section 4 `compute_transmission_angle()` works; plot renders with the workable band marked; in-band note is filled in
- [x] Section 5 has a motion artifact in `motion/` and the interference writeup
- [x] Section 6 hand calc covers three positions with working shown and a comparison note tying the points to the Section 3 plot
- [x] Section 7 triangulation summary and trust ledger are filled with specific entries; symmetry assumption appears as a known unknown
- [x] Section 8 reflection is filled in

> **Codespaces save warning.** Make sure the notebook is saved (`Ctrl+S`) and committed via Source Control before stopping your Codespace. Unstaged changes are lost when the container is deleted.

> **Connection to Part B.** Your design summary (Section 1) and trust ledger (Section 7) are the input artifacts your team uses on day one of Part B for the linkage comparison. Make them readable standalone — your teammates won't have this notebook open.
